In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import joblib
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded!")

✅ Libraries loaded!


In [20]:
# Load raw data
df = pd.read_csv('../data/raw/Food_waste.csv')

# Rename columns
df.columns = [
    'country', 'combined_kg_per_capita', 'household_kg_per_capita',
    'household_tonnes', 'retail_kg_per_capita', 'retail_tonnes',
    'foodservice_kg_per_capita', 'foodservice_tonnes',
    'confidence', 'M49_code', 'region', 'source'
]

print("✅ Data loaded!")
print("Shape:", df.shape)

✅ Data loaded!
Shape: (214, 12)


In [21]:
# Drop irrelevant columns
drop_cols = ['country', 'source', 'M49_code']
df_model = df.drop(columns=drop_cols)

# Our NEW target = real measured food waste per person
# This is NOT derived from other columns — it's a real measurement!
X = df_model.drop(columns=['combined_kg_per_capita'])
y = df_model['combined_kg_per_capita']

print("✅ Features and target defined!")
print("Target: combined_kg_per_capita (real measured value)")
print("Features:", X.columns.tolist())
print("X shape:", X.shape)
print("y shape:", y.shape)

✅ Features and target defined!
Target: combined_kg_per_capita (real measured value)
Features: ['household_kg_per_capita', 'household_tonnes', 'retail_kg_per_capita', 'retail_tonnes', 'foodservice_kg_per_capita', 'foodservice_tonnes', 'confidence', 'region']
X shape: (214, 8)
y shape: (214,)


In [22]:
le_confidence = LabelEncoder()
le_region     = LabelEncoder()

X['confidence_encoded'] = le_confidence.fit_transform(X['confidence'])
X['region_encoded']     = le_region.fit_transform(X['region'])

joblib.dump(le_confidence, '../models/le_confidence.pkl')
joblib.dump(le_region,     '../models/le_region.pkl')

X = X.drop(columns=['confidence', 'region'])

print("✅ Encoded!")
print("Final features:", X.columns.tolist())

✅ Encoded!
Final features: ['household_kg_per_capita', 'household_tonnes', 'retail_kg_per_capita', 'retail_tonnes', 'foodservice_kg_per_capita', 'foodservice_tonnes', 'confidence_encoded', 'region_encoded']


In [23]:

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

joblib.dump(scaler, '../models/scaler.pkl')

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print("✅ Scaled and split!")
print(f"Train: {X_train.shape[0]} samples")
print(f"Test : {X_test.shape[0]} samples")

✅ Scaled and split!
Train: 171 samples
Test : 43 samples


In [24]:
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv',   index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv',   index=False)

print("✅ Saved!")
print("\nFinal Feature List:")
for col in X.columns:
    print(f"   - {col}")
print(f"\nTarget: combined_kg_per_capita")

✅ Saved!

Final Feature List:
   - household_kg_per_capita
   - household_tonnes
   - retail_kg_per_capita
   - retail_tonnes
   - foodservice_kg_per_capita
   - foodservice_tonnes
   - confidence_encoded
   - region_encoded

Target: combined_kg_per_capita
